# Notebook 13: Alternative Dataset Integration (COVID-19 Radiography Database)
# for Cross-Dataset Generalization Testing

## Objective
Download and prepare the COVID-19 Radiography Database as a held-out,
cross-dataset generalization test set. This dataset provides an explicit
"Viral Pneumonia" category, useful for testing how well a Kermany-trained
model generalizes to viral pneumonia cases from a different source/population
-- directly relevant since VIRUS was the weakest class across all models in
our earlier results.

## Folder structure (fixed, do not change)
G:\Research paper\Research paper\Pneumonia-MultiModel-XAI\data set\
    raw\        <- original downloaded dataset goes here
    processed\  <- cleaned, ImageFolder-ready binary dataset goes here

In [1]:
# ============================================================
# Cell 2: Imports and Configuration
# ============================================================
import shutil
from pathlib import Path
from PIL import Image
from tqdm import tqdm

PROJECT_ROOT = Path("/mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI")

DATA_ROOT = PROJECT_ROOT / "data set"
RAW_DIR = DATA_ROOT / "raw" / "COVID-19_Radiography_Dataset"
PROCESSED_DIR = DATA_ROOT / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 260

print("Cell 2 : Config Set")
print(f"Raw data dir       : {RAW_DIR}")
print(f"Processed data dir : {PROCESSED_DIR}")

Cell 2 : Config Set
Raw data dir       : /mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI/data set/raw/COVID-19_Radiography_Dataset
Processed data dir : /mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI/data set/processed


In [2]:
# ============================================================
# Cell 3: Dataset Objective & Label Mapping Rules
# ============================================================
"""
Objective:
Prepare the COVID-19 Radiography Database as an external cross-dataset 
evaluation set to test binary generalization (NORMAL vs PNEUMONIA).

Label Mapping Strategy:
- Normal           -> NORMAL
- Viral Pneumonia  -> PNEUMONIA
- Lung_Opacity     -> PNEUMONIA
- COVID-19         -> Excluded
"""

print("Cell 3 : Label Mapping Rules Defined")
print("NORMAL    <- Normal")
print("PNEUMONIA <- Viral Pneumonia + Lung_Opacity")

Cell 3 : Label Mapping Rules Defined
NORMAL    <- Normal
PNEUMONIA <- Viral Pneumonia + Lung_Opacity


In [3]:
# ============================================================
# Cell 4: Download Dataset via Kaggle API (Optional / Verification)
# ============================================================
import subprocess

dataset_slug = "tawsifurrahman/covid19-radiography-database"

if RAW_DIR.exists() and any(RAW_DIR.iterdir()):
    print(f"Dataset already exists at: {RAW_DIR}. Skipping download.")
else:
    print(f"Downloading {dataset_slug}...")
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "kaggle", "datasets", "download",
        "-d", dataset_slug,
        "-p", str(RAW_DIR),
        "--unzip"
    ], check=True)
    print("Download and extraction complete.")

Dataset already exists at: /mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI/data set/raw/COVID-19_Radiography_Dataset. Skipping download.


In [4]:
# ============================================================
# Cell 5: Inspect Downloaded Folder Structure
# ============================================================
for item in RAW_DIR.rglob("*"):
    if item.is_dir():
        print(item.relative_to(RAW_DIR))

Lung_Opacity
Normal
Viral Pneumonia
Lung_Opacity/images
Lung_Opacity/masks
Normal/images
Normal/masks
Viral Pneumonia/images
Viral Pneumonia/masks


In [5]:
# ============================================================
# Cell 6: Map Classes to Binary NORMAL / PNEUMONIA and Organize
# ============================================================
def resolve_source_folder(base_folder_name):
    """Resolves whether to use 'images' subfolder or root folder to prevent duplicates."""
    target_path = RAW_DIR / base_folder_name
    images_subfolder = target_path / "images"
    if images_subfolder.exists():
        return images_subfolder
    elif target_path.exists():
        return target_path
    return None

SOURCE_DIRS = {
    "NORMAL": [
        p for p in [resolve_source_folder("Normal"), resolve_source_folder("normal")] if p
    ],
    "PNEUMONIA": [
        p for p in [
            resolve_source_folder("Viral Pneumonia"),
            resolve_source_folder("viral"),
            resolve_source_folder("Lung_Opacity"),
            resolve_source_folder("lung_opacity")
        ] if p
    ]
}

for class_name in ["NORMAL", "PNEUMONIA"]:
    (PROCESSED_DIR / class_name).mkdir(parents=True, exist_ok=True)

def process_and_copy(src_folder, dest_folder, prefix):
    if not src_folder or not src_folder.exists():
        return 0
    
    count = 0
    valid_extensions = {".png", ".jpg", ".jpeg"}
    # Case-insensitive extension search using iterdir()
    images = [f for f in src_folder.iterdir() if f.suffix.lower() in valid_extensions]
    
    for img_path in tqdm(images, desc=f"Processing {prefix}"):
        try:
            with Image.open(img_path) as img:
                img_rgb = img.convert("RGB")
                img_resized = img_rgb.resize((IMAGE_SIZE, IMAGE_SIZE))
                out_name = f"{prefix}_{img_path.stem}.png"
                img_resized.save(dest_folder / out_name)
                count += 1
        except Exception:
            continue
    return count

normal_count = 0
for src in SOURCE_DIRS["NORMAL"]:
    normal_count += process_and_copy(src, PROCESSED_DIR / "NORMAL", "normal")

pneumonia_count = 0
for src in SOURCE_DIRS["PNEUMONIA"]:
    prefix_name = src.parent.name if src.name == "images" else src.name
    pneumonia_count += process_and_copy(
        src, PROCESSED_DIR / "PNEUMONIA", prefix_name.replace(" ", "_").lower()
    )

print(f"\nNORMAL images processed    : {normal_count}")
print(f"PNEUMONIA images processed : {pneumonia_count}")

# Explicit validation check to prevent silent failures
if normal_count == 0 or pneumonia_count == 0:
    raise ValueError(
        f"Dataset processing failed! Found NORMAL: {normal_count}, PNEUMONIA: {pneumonia_count}. "
        "Check source folder paths."
    )

Processing lung_opacity: 100%|██████████| 6012/6012 [02:16<00:00, 44.01it/s]


NORMAL images processed    : 20384
PNEUMONIA images processed : 13369


In [6]:
# ============================================================
# Cell 7: Verify Processed Dataset Summary
# ============================================================
print("=" * 70)
print("Cell 7 : Processed Dataset Summary")
print("-" * 70)

for class_name in ["NORMAL", "PNEUMONIA"]:
    count = len(list((PROCESSED_DIR / class_name).glob("*.png")))
    print(f"{class_name:<12} : {count} images")

print("=" * 70)
print(f"Saved to : {PROCESSED_DIR}")

Cell 7 : Processed Dataset Summary
----------------------------------------------------------------------
NORMAL       : 10192 images
PNEUMONIA    : 7357 images
Saved to : /mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI/data set/processed
